# Deploy MiniMax-M2.5 on Azure ML — ND A100 80GB (NVLink, DP8+EP)

**Model:** `MiniMaxAI/MiniMax-M2.5` (230B total, 10B activated MoE)  
**VM:** `Standard_ND96amsr_A100_v4` — 8× NVIDIA A100 80GB SXM (NVLink 3.0)  
**Region:** West Europe  
**Serving:** vLLM with DP8+EP (data-parallel-size 8 + expert-parallel)  
**Image:** `vllm/vllm-openai:latest`

**Compared to the Poland NC A100 PCIe deployment:**
- NVLink enables GPU P2P → no `NCCL_P2P_DISABLE`, no `--enforce-eager`
- DP8+EP instead of TP4 → 8× data parallelism, no all-reduce overhead
- Still needs `VLLM_DISABLED_KERNELS=TritonFp8BlockScaledMMKernel` (A100 = sm_80)

**Expected improvement:** ~8-12× throughput over the PCIe setup.

This notebook orchestrates the full deployment lifecycle:
1. Install & import dependencies
2. Connect to AML workspace (West Europe)
3. Configuration
4. Build Docker image in ACR
5. Create managed online endpoint
6. Attach ACR to workspace (one-time setup)
7. Deploy MiniMax-M2.5 with DP8+EP
8. Set traffic to 100%
9. Get endpoint credentials
10. Test — chat completion & streaming
11. Performance benchmark
12. Check deployment logs
13. Cleanup (optional)

## 1. Install & Import Dependencies

In [ ]:
%pip install azure-ai-ml azure-identity openai python-dotenv --quiet

In [ ]:
import json
import os
import subprocess
import requests
from dotenv import load_dotenv
from azure.ai.ml import MLClient, load_online_endpoint, load_online_deployment
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

load_dotenv()

## 2. Connect to AML Workspace (West Europe)

In [ ]:
# Uses the West Europe workspace (_WE suffix in .env)
required_vars = ["AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP_WE", "AZURE_WORKSPACE_NAME_WE", "AZURE_TENANT_ID"]
missing = [v for v in required_vars if not os.environ.get(v)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}\nUpdate .env with West Europe workspace config")

subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
resource_group = os.environ["AZURE_RESOURCE_GROUP_WE"]
workspace_name = os.environ["AZURE_WORKSPACE_NAME_WE"]
tenant_id = os.environ["AZURE_TENANT_ID"]

try:
    credential = DefaultAzureCredential(additionally_allowed_tenants=[tenant_id])
    credential.get_token("https://management.azure.com/.default", tenant_id=tenant_id)
except Exception:
    credential = InteractiveBrowserCredential(tenant_id=tenant_id)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)
print(f"Connected to workspace: {ml_client.workspace_name} (region: West Europe)")

## 3. Configuration

In [ ]:
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME_ND", "minimax-m25-nd-prod")
DEPLOYMENT_NAME = os.environ.get("DEPLOYMENT_NAME", "current")
MODEL_NAME = os.environ.get("MODEL_NAME", "MiniMaxAI/MiniMax-M2.5")
CONFIG_DIR = "config_nda100-minimax25"

print(f"Endpoint:  {ENDPOINT_NAME}")
print(f"Model:     {MODEL_NAME}")
print(f"Config:    {CONFIG_DIR}/")

## 4. Build Docker Image in ACR

In [ ]:
ws = ml_client.workspaces.get(workspace_name)
acr_id = ws.container_registry

if not acr_id:
    result = subprocess.run(
        f'az acr list --resource-group {resource_group} --query "[].name" -o json',
        capture_output=True, text=True, shell=True,
    )
    acrs = json.loads(result.stdout) if result.returncode == 0 else []
    if acrs:
        acr_name = acrs[0]
        print(f"Workspace has no ACR. Using ACR from resource group: {acr_name}")
    else:
        acr_name = workspace_name.replace("-", "").replace("_", "").lower() + "acr"
        print(f"No ACR found. Creating '{acr_name}' in {resource_group}...")
        result = subprocess.run(
            f'az acr create --name {acr_name} --resource-group {resource_group} --sku Basic --admin-enabled true',
            capture_output=True, text=True, shell=True,
        )
        if result.returncode != 0:
            print("STDERR:", result.stderr[-2000:])
            raise RuntimeError("ACR creation failed")
        print(f"ACR '{acr_name}' created.")
else:
    acr_name = acr_id.split("/")[-1]

print(f"Using ACR: {acr_name}")

image_tag = f"{acr_name}.azurecr.io/minimax-m25-vllm:latest"
cmd = f'az acr build --registry {acr_name} --image minimax-m25-vllm:latest ./{CONFIG_DIR}'
print(f"Building image: {image_tag}")
print("This may take a few minutes...")

result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError(f"ACR build failed (exit code {result.returncode})")

print(f"Image built successfully: {image_tag}")

## 5. Create Managed Online Endpoint

In [ ]:
endpoint = load_online_endpoint(source=f"{CONFIG_DIR}/endpoint.yml")
endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint '{endpoint.name}' created (provisioning state: {endpoint.provisioning_state})")
print(f"Scoring URI: {endpoint.scoring_uri}")

## 6. Attach ACR to Workspace (one-time setup)

> **Run this cell only once** when setting up a new workspace/ACR combination. Skip on subsequent deployments.

In [ ]:
acr_resource_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.ContainerRegistry/registries/{acr_name}"
cmd = (
    f'az ml workspace update'
    f' --name {workspace_name}'
    f' --resource-group {resource_group}'
    f' --container-registry {acr_resource_id}'
    f' --update-dependent-resources'
)
result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])
    raise RuntimeError("Failed to attach ACR to workspace")

print(f"ACR '{acr_name}' attached to workspace '{workspace_name}'")

print("Deleting and recreating endpoint to refresh identity permissions...")
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print("Old endpoint deleted. Re-run cell 5 to recreate.")

## 7. Deploy MiniMax-M2.5 with DP8+EP

Deploys on `Standard_ND96amsr_A100_v4` (8× A100 80GB SXM, NVLink). Liveness/readiness probes have **1800s (30 min) initial delay** for model download + GPU loading. Takes **30-60+ minutes**.

In [ ]:
deployment = load_online_deployment(source=f"{CONFIG_DIR}/deployment.yml")
deployment.environment.image = image_tag

print(f"Starting deployment '{deployment.name}' on {deployment.instance_type}...")
print(f"Image: {image_tag}")
print(f"vLLM args: DP8+EP (data-parallel-size 8 + expert-parallel)")
print("This will take 30-60+ minutes (image pull + model download + GPU loading)")
deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment provisioning state: {deployment.provisioning_state}")

## 8. Set Traffic to 100%

In [ ]:
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
endpoint.traffic = {DEPLOYMENT_NAME: 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Traffic set to 100% on '{DEPLOYMENT_NAME}'")

## 9. Get Endpoint Credentials

In [ ]:
endpoint = ml_client.online_endpoints.get(ENDPOINT_NAME)
keys = ml_client.online_endpoints.get_keys(ENDPOINT_NAME)

scoring_uri = endpoint.scoring_uri
api_key = keys.primary_key

print(f"Scoring URI: {scoring_uri}")
print(f"API Key:     {api_key[:8]}...")

## 10. Test — Chat Completion

In [ ]:
base_url = scoring_uri.rstrip("/")
url = f"{base_url}/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}",
}
payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Explain what a Mixture-of-Experts model is in 3 sentences."},
    ],
    "max_tokens": 256,
}

response = requests.post(url, headers=headers, json=payload)
result = response.json()
print(json.dumps(result, indent=2))

## 10b. Test — Streaming with OpenAI SDK

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{base_url}/v1", api_key=api_key)

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Write a Python function to compute the Fibonacci sequence using memoization."},
    ],
    max_tokens=512,
    stream=True,
)

for chunk in response:
    delta = chunk.choices[0].delta
    if hasattr(delta, "content") and delta.content:
        print(delta.content, end="", flush=True)
print()

## 11. Performance Benchmark

Same benchmark as the Poland deployment — compare results to measure the ND A100 NVLink + DP8+EP improvement.

In [ ]:
import asyncio
import time
import random
import statistics
from openai import AsyncOpenAI

NUM_PROMPTS = 25
WARMUP = 5
MAX_CONCURRENCY = 10  # higher for DP8+EP (8 parallel streams)
MAX_OUTPUT_TOKENS = 256

WORD_POOL = "the quick brown fox jumps over a lazy dog while running through fields of green grass under bright blue skies with warm summer winds blowing gently across the open meadow".split()

def make_prompt(target_tokens=2048):
    words = [random.choice(WORD_POOL) for _ in range(target_tokens)]
    return f"Summarize the following text in one paragraph:\n\n{' '.join(words)}"

async def bench_one(client, prompt_id, prompt):
    t0 = time.perf_counter()
    ttft = None
    token_times = []
    usage_tokens = None

    stream = await client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_OUTPUT_TOKENS,
        stream=True,
        stream_options={"include_usage": True},
    )
    async for chunk in stream:
        now = time.perf_counter()
        if chunk.choices and chunk.choices[0].delta.content:
            if ttft is None:
                ttft = now - t0
            token_times.append(now)
        if hasattr(chunk, 'usage') and chunk.usage is not None:
            usage_tokens = chunk.usage.completion_tokens

    total_time = time.perf_counter() - t0
    generated_tokens = usage_tokens if usage_tokens else len(token_times)

    itls = [(token_times[i] - token_times[i-1]) * 1000 for i in range(1, len(token_times))]
    tpot = None
    if ttft and generated_tokens > 1:
        tpot = (total_time - ttft) / (generated_tokens - 1) * 1000

    return {"id": prompt_id, "ttft_ms": ttft * 1000 if ttft else None, "total_s": total_time, "tokens": generated_tokens, "tpot_ms": tpot, "itls": itls}

async def run_benchmark():
    client = AsyncOpenAI(base_url=f"{base_url}/v1", api_key=api_key)
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    prompts = [make_prompt() for _ in range(NUM_PROMPTS)]

    async def limited(i, p):
        async with semaphore:
            try:
                return await bench_one(client, i, p)
            except Exception as e:
                return {"id": i, "error": str(e)}

    print(f"Benchmarking: {NUM_PROMPTS} prompts ({WARMUP} warmup + {NUM_PROMPTS - WARMUP} measured)")
    print(f"  Concurrency: {MAX_CONCURRENCY}, Max output tokens: {MAX_OUTPUT_TOKENS}")
    print("Running...")

    t_start = time.perf_counter()
    results = await asyncio.gather(*[limited(i, p) for i, p in enumerate(prompts)])
    t_total = time.perf_counter() - t_start

    all_ok = sorted([r for r in results if "error" not in r], key=lambda r: r["id"])
    failed = [r for r in results if "error" in r]
    measured = all_ok[WARMUP:]

    ttfts = [r["ttft_ms"] for r in measured if r["ttft_ms"] is not None]
    tpots = [r["tpot_ms"] for r in measured if r["tpot_ms"] is not None]
    all_itls = [itl for r in measured for itl in r["itls"]]
    total_output_tokens = sum(r["tokens"] for r in measured)

    def pstats(name, values):
        if len(values) < 2:
            print(f"\n  --- {name}: insufficient data ({len(values)} samples) ---")
            return
        values.sort()
        print(f"\n  --- {name} (n={len(values)}) ---")
        print(f"  Mean: {statistics.mean(values):.0f} ms | Median: {values[len(values)//2]:.0f} ms | Stdev: {statistics.stdev(values):.0f} ms")
        print(f"  P99:  {values[min(int(len(values)*0.99), len(values)-1)]:.0f} ms | Min: {values[0]:.0f} ms | Max: {values[-1]:.0f} ms")

    print(f"\n{'='*60}")
    print(f"  ND A100 80GB NVLink — DP8+EP Benchmark Results")
    print(f"{'='*60}")
    print(f"  Successful: {len(measured)} | Failed: {len(failed)} | Duration: {t_total:.1f}s")
    print(f"  Total tokens: {total_output_tokens} | Throughput: {total_output_tokens/t_total:.1f} tok/s | {len(measured)/t_total:.2f} req/s")
    pstats("TTFT", ttfts)
    pstats("TPOT", tpots)
    pstats("ITL", all_itls)
    if failed:
        print(f"\n  Errors: {[r['error'][:100] for r in failed[:3]]}")
    print(f"\n  Token counting: server-reported usage in {sum(1 for r in measured if r['tokens'] > 0)}/{len(measured)} responses")

await run_benchmark()

## 12. Check Deployment Logs

In [ ]:
logs = ml_client.online_deployments.get_logs(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    lines=100,
)
print(logs)

## 13. Cleanup (Optional)

**Only run this when you're done with the POC.** This deletes the endpoint and deployment to stop billing.

In [ ]:
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print(f"Endpoint '{ENDPOINT_NAME}' deleted.")